In [1]:
import glob
import json
import os
import re
import numpy as np
import pandas as pd

# ==================== 1. 設定固定權威 INTRA 實驗資料夾對照表 ====================
INTRA_EXPERIMENT_CONFIGS = [
    {
        "folder": (
            "social-simulate-v3-agent-qwen3-authority-auth_qwen3_8b-intra-result"
        ),
        "family": "Qwen3 Family (INTRA)",
        "condition": "Authority (8B) + Memory/Reflection",
        "auth_scale": "8B",
        "has_authority": True,
    },
    {
        "folder": (
            "social-simulate-v3-agent-qwen3-authority-auth_qwen3_14b-intra-result"
        ),
        "family": "Qwen3 Family (INTRA)",
        "condition": "Authority (14B) + Memory/Reflection",
        "auth_scale": "14B",
        "has_authority": True,
    },
    {
        "folder": (
            "social-simulate-v3-agent-qwen3-authority-auth_qwen3_30b-intra-result"
        ),
        "family": "Qwen3 Family (INTRA)",
        "condition": "Authority (30B) + Memory/Reflection",
        "auth_scale": "30B",
        "has_authority": True,
    },
    {
        "folder": "social-simulate-v3-agent-qwen3-control-intra-result",
        "family": "Qwen3 Family (INTRA)",
        "condition": "Control (No Authority) + Memory/Reflection",
        "auth_scale": "N/A",
        "has_authority": False,
    },
]

MODEL_SCALE_MAP = {
    "qwen3:8b": "8B",
    "qwen3:14b": "14B",
    "qwen3:30b": "30B",
    "llama3:8b": "8B",
    "deepseek-r1:8b": "8B",
}


def clean_model_name(raw_name):
    """將原生名稱轉換為論文簡潔標籤 8B/14B/30B"""
    raw_lower = str(raw_name).lower()
    for key, label in MODEL_SCALE_MAP.items():
        if key in raw_lower:
            return label
    match = re.search(r"(\d+b)", raw_lower)
    return match.group(1).upper() if match else raw_name


def parse_intra_run_details(df):
    """深度拆解每一場 Run 的 agent_details_json，提煉出微觀行為與精確權威引導成功率"""
    micro_records = []
    macro_auth_success = []

    for run_idx, row in df.iterrows():
        json_str = row.get("agent_details_json", "{}")
        try:
            details = json.loads(json_str)
        except Exception:
            continue

        auth_name = None
        auth_init_stance = None
        auth_model_scale = None

        # 1. 先定位權威角色
        for name, info in details.items():
            if info.get("is_authority", False):
                auth_name = name
                auth_init_stance = info.get("initial_stance", "")
                auth_model_scale = clean_model_name(
                    info.get("model_name", "Unknown")
                )
                break

        aligned_count = 0
        non_auth_count = 0

        # 2. 計算一般 Agent 行為
        for name, info in details.items():
            is_auth = info.get("is_authority", False)
            model_scale = clean_model_name(info.get("model_name", "Unknown"))
            init_s = float(info.get("initial_score", 5.0))
            final_s = float(info.get("final_score", 5.0))
            shift = final_s - init_s
            shift_type = info.get("shift_type", "堅持")

            swayed_towards_auth = False
            if auth_name and not is_auth:
                non_auth_count += 1

                # 💡 關鍵修正：權威引導成功 (Stance Alignment) 的正確判定
                # 情況 A：權威立場為 Support，若 Agent 最終立場在支持陣營 (final_s >= 6.0) 或顯著向權威移動 (shift > 0)
                if auth_init_stance == "Support":
                    if final_s >= 6.0 or shift > 0:
                        swayed_towards_auth = True
                        aligned_count += 1
                # 情況 B：權威立場為 Oppose，若 Agent 最終立場在反對陣營 (final_s <= 4.0) 或顯著向權威移動 (shift < 0)
                elif auth_init_stance == "Oppose":
                    if final_s <= 4.0 or shift < 0:
                        swayed_towards_auth = True
                        aligned_count += 1

            micro_records.append({
                "run_id": row.get("run_id", run_idx + 1),
                "agent_name": name,
                "model_scale": model_scale,
                "is_authority": is_auth,
                "initial_score": init_s,
                "final_score": final_s,
                "stance_shift": shift,
                "shift_type": shift_type,
                "swayed_towards_auth": (
                    swayed_towards_auth if not is_auth else np.nan
                ),
                "auth_model_scale_of_run": auth_model_scale,
            })

        if non_auth_count > 0:
            macro_auth_success.append(aligned_count / non_auth_count)

    macro_avg_success = (
        np.mean(macro_auth_success) * 100 if macro_auth_success else np.nan
    )
    return pd.DataFrame(micro_records), macro_avg_success


def process_intra_folder(folder_path, has_authority, condition_name, auth_scale):
    """處理單一實驗資料夾，計算 7 大巨觀指標與微觀拆解"""
    csv_pattern = os.path.join(folder_path, "*_metrics.csv")
    csv_files = glob.glob(csv_pattern)

    if not csv_files:
        print(f"⚠️ 警告：在路徑 '{folder_path}' 中找不到 *_metrics.csv 檔案。")
        return None, None

    target_csv = sorted(csv_files)[-1]
    df = pd.read_csv(target_csv)

    micro_df, macro_auth_rate = parse_intra_run_details(df)

    # 1. 巨觀 7 大指標 (Macro Summary)
    macro_summary = {
        "runs_count": len(df),
        "auth_scale": auth_scale,
        "mean_shift": (
            round(df["mean_shift"].mean(), 3) if "mean_shift" in df else np.nan
        ),
        "polarization_index": (
            round(df["polarization_index"].mean(), 3)
            if "polarization_index" in df
            else np.nan
        ),
        "fragmentation_index": (
            round(df["fragmentation_index"].mean(), 3)
            if "fragmentation_index" in df
            else np.nan
        ),
        "conformity_rate": (
            round(df["conformity_rate"].mean(), 3)
            if "conformity_rate" in df
            else np.nan
        ),
        "avg_authority_in_degree": (
            round(df["authority_in_degree"].mean(), 2)
            if "authority_in_degree" in df
            else 0.0
        ),
        "mean_cascade_depth": (
            round(df["cascade_depth"].mean(), 2)
            if "cascade_depth" in df
            else 0.0
        ),
        "authority_success_rate": (
            round(macro_auth_rate, 2)
            if (has_authority and not np.isnan(macro_auth_rate))
            else "N/A (控制組)"
        ),
    }

    # 2. 微觀模型規模拆解 (表 4-8 專用)
    micro_breakdown_list = []
    if micro_df is not None and not micro_df.empty:
        for scale in ["8B", "14B", "30B"]:
            non_auth_sub = micro_df[
                (micro_df["model_scale"] == scale)
                & (micro_df["is_authority"] == False)
            ]

            if not non_auth_sub.empty:
                avg_shift = round(non_auth_sub["stance_shift"].mean(), 3)
                sway_rate = (
                    round(non_auth_sub["swayed_towards_auth"].mean() * 100, 2)
                    if has_authority
                    else "N/A"
                )
                persist_rate = round(
                    (non_auth_sub["shift_type"] == "堅持").mean() * 100, 2
                )

                micro_breakdown_list.append({
                    "Condition": condition_name,
                    "Auth_Scale": auth_scale,
                    "Agent_Scale": scale,
                    "Mean_Stance_Shift": avg_shift,
                    "Persistence_Rate(%)": persist_rate,
                    "Swayed_By_Authority_Rate(%)": sway_rate,
                })

    return macro_summary, pd.DataFrame(micro_breakdown_list)


def main():
    print("=" * 90)
    print(" 🚀 開始執行數據整合分析 (對齊 7 大標準欄位)")
    print("=" * 90 + "\n")

    macro_results = []
    micro_all_results = []

    for config in INTRA_EXPERIMENT_CONFIGS:
        folder = config["folder"]
        family = config["family"]
        condition = config["condition"]
        auth_scale = config["auth_scale"]
        has_auth = config["has_authority"]

        print(f"🔍 正在處理：[{family}] - [{condition}]...")
        macro_sum, micro_df = process_intra_folder(
            folder, has_auth, condition, auth_scale
        )

        if macro_sum:
            macro_sum["Family"] = family
            macro_sum["Condition"] = condition
            macro_results.append(macro_sum)

        if micro_df is not None and not micro_df.empty:
            micro_all_results.append(micro_df)

    # ------------------ 輸出 1：7 大巨觀指標矩陣 ------------------
    if macro_results:
        macro_df = pd.DataFrame(macro_results)
        cols = [
            "Family",
            "Condition",
            "auth_scale",
            "runs_count",
            "mean_shift",
            "polarization_index",
            "fragmentation_index",
            "conformity_rate",
            "avg_authority_in_degree",
            "mean_cascade_depth",
            "authority_success_rate",
        ]
        macro_df = macro_df[cols].rename(columns={
            "Family": "模型家族",
            "Condition": "實驗條件",
            "auth_scale": "權威模型規模",
            "runs_count": "總模擬次數",
            "mean_shift": "平均立場位移 (Mean Shift)",
            "polarization_index": "群體極化指數 (Polarization)",
            "fragmentation_index": "群體分裂指數 (Fragmentation)",
            "conformity_rate": "從眾率 (Conformity Rate)",
            "avg_authority_in_degree": "權威被提及平均次數 (In-degree)",
            "mean_cascade_depth": "級聯擴散平均深度 (Cascade Depth)",
            "authority_success_rate": "權威引導成功率(%) (Stance Alignment)",
        })

        print("\n" + "=" * 100)
        print("📊 表 1：巨觀 7 大對比矩陣 (Macro Matrix)")
        print("=" * 100)
        print(macro_df.to_string(index=False))
        macro_df.to_csv(
            "macro_summary_7_metrics.csv", index=False, encoding="utf-8-sig"
        )

    # ------------------ 輸出 2：表 4-8 微觀模型拆解矩陣 ------------------
    if micro_all_results:
        combined_micro_df = pd.concat(micro_all_results, ignore_index=True)
        cols_micro = [
            "Condition",
            "Auth_Scale",
            "Agent_Scale",
            "Mean_Stance_Shift",
            "Persistence_Rate(%)",
            "Swayed_By_Authority_Rate(%)",
        ]
        combined_micro_df = combined_micro_df[cols_micro].rename(columns={
            "Condition": "實驗條件",
            "Auth_Scale": "權威模型規模",
            "Agent_Scale": "一般Agent模型規模",
            "Mean_Stance_Shift": "平均立場位移",
            "Persistence_Rate(%)": "立場堅持率(%)",
            "Swayed_By_Authority_Rate(%)": "被權威同化順服率(%)",
        })

        print("\n" + "=" * 100)
        print("🔬 表 4-8：不同權威規模下一般 Agent 之微觀反應拆解表")
        print("=" * 100)
        print(combined_micro_df.to_string(index=False))
        combined_micro_df.to_csv(
            "intra_micro_scale_breakdown_v3.csv",
            index=False,
            encoding="utf-8-sig",
        )


if __name__ == "__main__":
    main()

 🚀 開始執行數據整合分析 (對齊 7 大標準欄位)

🔍 正在處理：[Qwen3 Family (INTRA)] - [Authority (8B) + Memory/Reflection]...
🔍 正在處理：[Qwen3 Family (INTRA)] - [Authority (14B) + Memory/Reflection]...
🔍 正在處理：[Qwen3 Family (INTRA)] - [Authority (30B) + Memory/Reflection]...
🔍 正在處理：[Qwen3 Family (INTRA)] - [Control (No Authority) + Memory/Reflection]...

📊 表 1：巨觀 7 大對比矩陣 (Macro Matrix)
                模型家族                                       實驗條件 權威模型規模  總模擬次數  平均立場位移 (Mean Shift)  群體極化指數 (Polarization)  群體分裂指數 (Fragmentation)  從眾率 (Conformity Rate)  權威被提及平均次數 (In-degree)  級聯擴散平均深度 (Cascade Depth) 權威引導成功率(%) (Stance Alignment)
Qwen3 Family (INTRA)         Authority (8B) + Memory/Reflection     8B     25               -0.800                  0.480                   0.774                  0.294                  26.04                      1.96                          54.4
Qwen3 Family (INTRA)        Authority (14B) + Memory/Reflection    14B     25               -0.674                  0.392                   0.607